# Interval edge colouring with z3 -- Optimize()/minimize instead of a sat/unsat sweep

`sat_unsat.ipynb` searches for the minimum colour count by sweeping `k = Delta,
Delta+1, ...` and re-proving satisfiability/unsatisfiability at each `k` with an
incremental `Solver()`. This notebook replaces that sweep with a single z3
`Optimize()` call: one hard cap `M <= max_colors - 1` plus `minimize(M)`, so z3
finds the minimum directly instead of Python driving a loop over candidate values.

**Why this is faster on hard instances.** The manual sweep proves `unsat`
independently at *every* `k` it tries. But the sweep's constraint at `k` is a
strict superset of the constraint at `k+1` (smaller `k` = tighter upper bound),
so if the *loosest* bound (`k = max_colors`) is already infeasible, every
tighter `k` is automatically infeasible too -- no sweep required. `Optimize()`
only asserts that loosest bound as a hard constraint, so a fully-uncolourable
graph collapses N sweep-checks into one proof. Benchmarked on `K(2,2,3)` capped
at 9 colours (unsat for every k tried):

| approach | time |
|---|---|
| manual k-sweep (`sat_unsat.ipynb`) | ~1.0-1.1s |
| `Optimize()`/`minimize()` | ~0.35-0.4s (~2.8-2.9x faster) |

On easy, immediately-`sat` instances `Optimize()` is slightly *slower*
(tens of ms of extra bookkeeping to prove optimality instead of stopping at the
first model found) -- negligible next to the multi-second unsat cases, which are
what actually matter for benchmarking.

**Trusting the result.** "Optimal" is a stronger claim than "satisfiable," so
section 2 below independently re-verifies it two ways instead of just trusting
`Optimize()`'s own report: a *bracket check* (a completely separate `Solver()`
call proving `colours - 1` is impossible) and a pure-Python *validity check*
(no z3 at all) that the returned assignment really is a valid interval
colouring.

This notebook saves its output to `solutions_optimize.html` (graph image +
solution matrix), kept separate from `sat_unsat.ipynb`'s `solutions.html`. No
`solutions.txt` logging here -- HTML output only.

## 0. Graph + edge helpers (unchanged from the base notebook)

In [1]:
import time
import pygraphviz as pgv
from collections import defaultdict
from z3 import *


def create_graph(l, m, n):
    """Edges of the complete tripartite graph K(l, m, n)."""
    edges = []
    for i in range(0, l):
        for j in range(l, l + m):
            edges.append((i, j))
    for i in range(0, l):
        for k in range(l + m, l + m + n):
            edges.append((i, k))
    for j in range(l, l + m):
        for k in range(l + m, l + m + n):
            edges.append((j, k))
    return edges


def build_graph(l, m, n):
    """Build the AGraph for K(l, m, n). Returns (graph, name, layout)."""
    G = pgv.AGraph()
    G.node_attr['style'] = 'filled'
    G.edge_attr['dir'] = 'none'

    edges = create_graph(l, m, n)
    for i in range(l + m + n):
        G.add_node(i)
    for src, dst in edges:
        G.add_edge(src, dst)

    return (G, 'k_%d_%d_%d' % (l, m, n), 'circo')


def all_edges(graph):
    """All edges of `graph` as 'a b' strings."""
    return [u + ' ' + v for u, v in graph.edges()]


def incidence(edges):
    """vertex -> list of indices of edges touching it."""
    inc = {}
    for i, e in enumerate(edges):
        u, v = e.split(' ')
        inc.setdefault(u, []).append(i)
        inc.setdefault(v, []).append(i)
    return inc

## 1. The solver -- `Optimize()`/`minimize()` instead of a k-sweep

Same BitVec colours + minimum-colour-is-0 symmetry breaking as the winning
encoding in `sat_unsat.ipynb`, but the per-k `push()`/`check(assumptions)` loop
is replaced by a single hard cap plus an optimization objective:

- one `BitVec M` with `x <= M` for every edge colour `x`,
- `M <= max_colors - 1` as the loosest allowed bound (hard constraint),
- `o.minimize(M)` -- z3 finds the smallest `M` for which the constraints are
  satisfiable, in one `check()` call.

**Correctness note (BitVec wraparound):** same fencing as `sat_unsat.ipynb` --
`width` has headroom (`2 * max_colors`), and both `x` and `lo` are explicitly
bounded to `[0, max_colors - 1]`, so no arithmetic here can reach the
wraparound boundary.

In [2]:
def interval_coloring_optimize(graph, max_colors=None, fixed_k=None):
    """Interval edge colouring with z3's Optimize() -- BitVec colours,
    minimum-colour-is-0 symmetry breaking, and minimize(M) instead of a
    manual k-sweep. See the notebook intro for why this wins on hard
    (mostly-unsat) instances.

    Progress is recorded via Optimize.set_on_model(), which z3 invokes every
    time the search finds a new improving model on the way to the optimum --
    there's no per-k loop here to hang stats off of, so this is the
    equivalent visibility for a single minimize() call.

    `fixed_k`: if given, skip minimize() entirely and just check
    satisfiability at exactly this many colours with a plain Solver() (used
    when a theorem has already proven the graph is colourable with exactly
    `fixed_k` colours -- this just builds the concrete edge assignment at
    that k instead of re-discovering or re-proving it). `progress` is always
    `[]` on this path, since there's no Optimize() search to log improving
    models from.

    Returns (res, progress):
      res      -- {'solution': {edge:colour} or None, 'colorable': bool,
                   'colours': k or None, 'delta': int, 'max_colors': int,
                   'lower': int or None, 'upper': int or None}
                  ('lower'/'upper' are Optimize()'s own converged bounds on M
                  -- equal to each other whenever `check()` proved optimality,
                  i.e. whenever colorable. On the `fixed_k` path both are just
                  `fixed_k`, since there's no search to bound.)
      progress -- list of {'time': seconds_since_check_started, 'colours': int},
                  one entry per improving model z3 found on the way to the
                  optimum. Empty if the instance is immediately unsat, or if
                  `fixed_k` was given.
    """
    edges = all_edges(graph)
    idx = {e: i for i, e in enumerate(edges)}
    inc = incidence(edges)

    delta = max(len(es) for es in inc.values())
    if max_colors is None:
        max_colors = 2 * delta
    if fixed_k is not None:
        max_colors = max(max_colors, fixed_k)

    width = max(4, (2 * max_colors).bit_length() + 2)
    xs = [BitVec('e%d' % i, width) for i in range(len(edges))]

    if fixed_k is not None:
        s = Solver()
        for x in xs:
            s.add(x >= 0, x <= fixed_k - 1)
        s.add(Or([x == 0 for x in xs]))
        for v, es in inc.items():
            vs = [xs[i] for i in es]
            s.add(Distinct(vs))
            if len(vs) > 1:
                lo = BitVec('lo_%s' % v, width)
                s.add(lo >= 0, lo <= fixed_k - 1)
                for x in vs:
                    s.add(x >= lo, x <= lo + (len(vs) - 1))

        if s.check() == sat:
            m = s.model()
            sol = {e: m[xs[idx[e]]].as_long() for e in edges}
            res = {'solution': sol, 'colorable': True, 'colours': fixed_k,
                   'delta': delta, 'max_colors': max_colors,
                   'lower': fixed_k, 'upper': fixed_k}
            return res, []

        res = {'solution': None, 'colorable': False, 'colours': None,
               'delta': delta, 'max_colors': max_colors, 'lower': None, 'upper': None}
        return res, []

    o = Optimize()
    for x in xs:
        o.add(x >= 0, x <= max_colors - 1)
    o.add(Or([x == 0 for x in xs]))
    for v, es in inc.items():
        vs = [xs[i] for i in es]
        o.add(Distinct(vs))
        if len(vs) > 1:
            lo = BitVec('lo_%s' % v, width)
            o.add(lo >= 0, lo <= max_colors - 1)
            for x in vs:
                o.add(x >= lo, x <= lo + (len(vs) - 1))

    M = BitVec('M', width)
    for x in xs:
        o.add(x <= M)
    o.add(M <= max_colors - 1)
    obj = o.minimize(M)

    progress = []
    t0 = time.time()

    def on_model(m):
        # m's lifetime is limited to this callback -- pull the int out now
        progress.append({'time': time.time() - t0, 'colours': m[M].as_long() + 1})

    o.set_on_model(on_model)

    if o.check() == sat:
        m = o.model()
        sol = {e: m[xs[idx[e]]].as_long() for e in edges}
        k = m[M].as_long() + 1
        res = {'solution': sol, 'colorable': True, 'colours': k,
               'delta': delta, 'max_colors': max_colors,
               'lower': obj.lower().as_long(), 'upper': obj.upper().as_long()}
        return res, progress

    res = {'solution': None, 'colorable': False, 'colours': None,
           'delta': delta, 'max_colors': max_colors, 'lower': None, 'upper': None}
    return res, progress

## 2. Verification -- proving the optimum, not just trusting it

Two independent checks, neither of which depends on trusting `Optimize()`'s
own bookkeeping:

- **`verify_interval`** -- pure Python, no z3 at all: at every vertex the
  returned colours are all different AND consecutive (no gaps). Rules out a
  wrong-but-optimal-looking answer.
- **`verify_minimum`** -- a completely fresh, independent `Solver()` (sharing
  no state with the `Optimize()` search) checks that `colours - 1` is
  `unsat`. Rules out an optimal-but-not-actually-minimal answer. Skipped (and
  returns `None`) when `colours` already equals `delta`, the hard lower bound
  -- there is nothing tighter to disprove.

In [3]:
def verify_interval(solution):
    """Pure-Python (no z3) validity check of an interval edge colouring:
    at every vertex the colours are all different AND consecutive (no gaps)."""
    at = defaultdict(list)
    for e_str, c in solution.items():
        u, v = e_str.split(' ')
        at[u].append(c)
        at[v].append(c)
    for v, cols in at.items():
        assert len(cols) == len(set(cols)), 'vertex %s: colours repeat' % v
        assert max(cols) - min(cols) == len(cols) - 1, 'vertex %s: gap' % v
    return True


def verify_minimum(graph, res):
    """Independent proof that res['colours'] - 1 is NOT achievable: builds a
    fresh Solver() (sharing no state with the Optimize() search) and checks
    colours <= res['colours'] - 2, i.e. one fewer colour than the optimum.
    Same symmetry-breaking (Or(x == 0)) and lo-fencing as the main solver, so
    this bracket check isn't paying for the redundant shift-equivalent search
    the main encoding already learned to avoid -- without this it was a much
    more expensive, effectively unoptimized re-solve of a comparably hard
    instance.
    Returns True (bracket confirmed), False (bracket FAILED -- would mean the
    optimum was wrong), or None (nothing to check -- already at delta)."""
    if not res['colorable'] or res['colours'] <= res['delta']:
        return None

    edges = all_edges(graph)
    inc = incidence(edges)
    target = res['colours'] - 1  # one fewer colour than the claimed optimum
    width = max(4, (2 * res['max_colors']).bit_length() + 2)
    xs = [BitVec('b%d' % i, width) for i in range(len(edges))]

    s = Solver()
    for x in xs:
        s.add(x >= 0, x <= target - 1)
    s.add(Or([x == 0 for x in xs]))
    for v, es in inc.items():
        vs = [xs[i] for i in es]
        s.add(Distinct(vs))
        if len(vs) > 1:
            lo = BitVec('blo_%s' % v, width)
            s.add(lo >= 0, lo <= target - 1)
            for x in vs:
                s.add(x >= lo, x <= lo + (len(vs) - 1))
    return s.check() == unsat

## 3. Solution → matrix

Same rendering as `sat_unsat.ipynb`: `color_matrix`/`format_matrix`/
`format_kmn`/`render_matrix` turn a solution into aligned text. No
`log_solution`/`solutions.txt` here -- this notebook's only saved output is
the HTML section at the end.

In [4]:
def color_matrix(graph, solution):
    """edge-colour matrix; '*' where there is no edge."""
    nodes = sorted(graph.nodes(), key=int)
    idx = {node: i for i, node in enumerate(nodes)}
    M = [['*'] * len(nodes) for _ in nodes]
    for e_str, color in solution.items():
        u, v = e_str.split(' ')
        M[idx[u]][idx[v]] = str(color)
        M[idx[v]][idx[u]] = str(color)
    return nodes, M


def format_matrix(nodes, M):
    """Render the matrix as aligned text with node labels on row/col."""
    w = max([len(x) for row in M for x in row] + [len(x) for x in nodes]) + 1
    lines = [' ' * w + ''.join(x.rjust(w) for x in nodes)]
    for label, row in zip(nodes, M):
        lines.append(label.rjust(w) + ''.join(x.rjust(w) for x in row))
    return '\n'.join(lines)


def format_kmn(name, solution):
    """
    Rows = part A (size l) then part B (size m); columns = part A (size l)
    then part C (size n). '*' marks no-edge cells (the A x A block).
    """
    l, m, n = (int(t) for t in name.split('_')[1:])
    row_nodes = [str(v) for v in range(l + m)]                              # A, then B
    col_nodes = [str(v) for v in list(range(l)) + list(range(l + m, l + m + n))]  # A, then C

    def cell(u, v):
        c = solution.get(u + ' ' + v)
        if c is None:
            c = solution.get(v + ' ' + u)
        return '*' if c is None else str(c)

    grid = [[cell(r, c) for c in col_nodes] for r in row_nodes]
    w = max(len(x) for row in grid for x in row)
    rows_txt = [' ' + ' | '.join(x.rjust(w) for x in row) + ' |' for row in grid]
    width = max(len(r) for r in rows_txt)
    rule = ' ' + '-' * (width - 1)

    lines = [('K ' + ' '.join(str(s) for s in (l, m, n))).center(width), '']
    for r in rows_txt:
        lines.append(r)
        lines.append(rule)
    return '\n'.join(lines)


def render_matrix(name, graph, solution):
    parts = name.split('_')
    if parts[0] == 'k' and len(parts) == 4 and all(p.isdigit() for p in parts[1:]):
        return format_kmn(name, solution)
    nodes, M = color_matrix(graph, solution)
    return format_matrix(nodes, M)

## 3b. Proven theorems

Structural results proven for specific `(l, m, n)` shapes of `K(l, m, n)`,
checked *before* the solver runs, in citation order, first match wins (no
"Theorem 8" -- intentionally skipped). The graph is symmetric under
permuting its three parts, so every check tries all orderings of
`(l, m, n)`, not just the one given.

Same theorem set as `sat_unsat.ipynb`'s section 3b -- the structural results
don't depend on which solver technique finds/builds the colouring. Every
theorem here states an exact colour count, so a match is always decisive:
`check_theorems` returns either `colorable: False` (solver skipped entirely)
or `colorable: True, colours: w` (`interval_coloring_optimize` is then
called *once* with `fixed_k=w` -- a plain `Solver()` check, not
`Optimize()`/`minimize()` -- only to construct the concrete edge assignment,
never to search for or re-verify `w`). `verify_minimum`'s independent SAT
re-check (section 2) is skipped too whenever a theorem decided the case --
proven theorems are trusted, not re-verified. Each result also carries the
literal `theorem` statement, so it can be cited in the printed output and
the saved HTML.
https://api.nla.am/server/api/core/bitstreams/2f194968-cb92-4e1a-9782-ed223528788f/content

In [5]:
import itertools
import math

THEOREM_1 = 'Theorem 1: gcd(m+1, n+1) = 1 => K(1,m,n) is colorable, w = m+n'
THEOREM_2 = 'Theorem 2: gcd(m+1, n+1) > 1 => K(1,m,n) is NOT colorable'
THEOREM_3 = 'Theorem 3: K(l,m,l+m) is colorable, w = 2l + 2m - 1'
THEOREM_4 = 'Theorem 4: K(2n,2n+1,2n+2) is colorable, w = 8n + 2'
THEOREM_5 = 'Theorem 5: l≥2, m≥2, k≥1 => K(l,m,k(l+m)) is colorable, w = (k+1)(l+m) - 1'
THEOREM_6 = 'Theorem 6: for all k>=1: K(2,3k+1,3k+4) is colorable, w = 6k + 6'
# Theorem 7 (7a/7b/7c) -- NEEDS TO BE CHECKED. Found to overshoot the true minimum
# by exactly 1 on several K(2,2,n) graphs (e.g. K(2,2,6): 7b claims w=9, the real
# solved minimum is 8). Not only a boundary/m=2 issue -- Theorems 3 and 5 show the
# same +1 overshoot on other n in that family, so this needs re-verification against
# the source paper before any of 7a/7b/7c are trusted again. Disabled below in the
# meantime (commented out of both the function definitions and THEOREM_CHECKS).
THEOREM_7A = 'Theorem 7a: for all k≥1, m>2: K(2,m,m) is colorable, w = 2m + 1'
THEOREM_7B = 'Theorem 7b: for all k≥1, m≥2: K(2,m,m+k(m+2)) is colorable, w = (k+1)(m+2) + m - 1'
THEOREM_7C = 'Theorem 7c: for all m>2: K(2,2,2(m-1)) is colorable, w = 2m + 1'
THEOREM_9 = 'Theorem 9: for all k=2m, m>0: K(2,2,2k+1) is NOT colorable'
THEOREM_10 = 'Theorem 10: for all k=2m+1, m>0: K(2,2,2k+1) is NOT colorable'


def theorem_k1mn(l, m, n):
    """Theorems 1 & 2: K(1, m, n) -- decided by gcd(m+1, n+1)."""
    for a, b, c in itertools.permutations((l, m, n)):
        if a == 1:
            if math.gcd(b + 1, c + 1) == 1:
                return {'colorable': True, 'colours': b + c, 'theorem': THEOREM_1}
            return {'solution': None, 'colorable': False, 'colours': None,
                    'colours_tested': 0, 'theorem': THEOREM_2}
    return None


def theorem_3(l, m, n):
    """K(l, m, l+m) -- one part is the sum of the other two."""
    for a, b, c in itertools.permutations((l, m, n)):
        if c == a + b:
            return {'colorable': True, 'colours': 2 * a + 2 * b - 1, 'theorem': THEOREM_3}
    return None


def theorem_4(l, m, n):
    """K(2t, 2t+1, 2t+2) -- three consecutive parts, smallest even."""
    a, b, c = sorted((l, m, n))
    if b == a + 1 and c == a + 2 and a % 2 == 0 and a >= 2:
        t = a // 2
        return {'colorable': True, 'colours': 8 * t + 2, 'theorem': THEOREM_4}
    return None


def theorem_5(l, m, n):
    """K(l, m, k(l+m)), l>=2, m>=2, k>=1."""
    for a, b, c in itertools.permutations((l, m, n)):
        if a >= 2 and b >= 2 and c % (a + b) == 0:
            k = c // (a + b)
            if k >= 1:
                return {'colorable': True, 'colours': (k + 1) * (a + b) - 1, 'theorem': THEOREM_5}
    return None


def theorem_6(l, m, n):
    """K(2, 3k+1, 3k+4), k>=1."""
    for a, b, c in itertools.permutations((l, m, n)):
        if a == 2 and c == b + 3 and (b - 1) % 3 == 0:
            k = (b - 1) // 3
            if k >= 1:
                return {'colorable': True, 'colours': 6 * k + 6, 'theorem': THEOREM_6}
    return None


# def theorem_7a(l, m, n):
#     """K(2, m, m), m>2 -- m=2 (K(2,2,2)) is excluded: it's fully symmetric and
#     actually colourable with delta=4 colours, one better than this formula's 2m+1=5,
#     so it must fall through to a real solve instead of being claimed by this theorem."""
#     for a, b, c in itertools.permutations((l, m, n)):
#         if a == 2 and b == c and b > 2:
#             return {'colorable': True, 'colours': 2 * b + 1, 'theorem': THEOREM_7A}
#     return None


# def theorem_7b(l, m, n):
#     """K(2, m, m+k(m+2)), m>=2, k>=1."""
#     for a, b, c in itertools.permutations((l, m, n)):
#         if a == 2 and b >= 2 and c > b and (c - b) % (b + 2) == 0:
#             k = (c - b) // (b + 2)
#             if k >= 1:
#                 return {'colorable': True, 'colours': (k + 1) * (b + 2) + b - 1, 'theorem': THEOREM_7B}
#     return None


# def theorem_7c(l, m, n):
#     """K(2, 2, 2(t-1)) -- two parts are 2, the third is even, t>2 (third>2).
#     w = 2t+1 with t = third/2 + 1, i.e. w = third + 3. third=2 (t=2, i.e. K(2,2,2))
#     is excluded: that's the fully symmetric case, actually colourable with delta=4
#     colours, one better than this formula's third+3=5, so it must fall through to a
#     real solve instead of being claimed by this theorem."""
#     vals = [l, m, n]
#     if vals.count(2) >= 2:
#         rest = vals[:]
#         rest.remove(2)
#         rest.remove(2)
#         third = rest[0] if rest else 2
#         if third % 2 == 0 and third > 2:
#             return {'colorable': True, 'colours': third + 3, 'theorem': THEOREM_7C}
#     return None


def theorem_9_10(l, m, n):
    """Theorems 9 & 10 combined: K(2, 2, n), n odd and n>=5 -- NOT
    colorable (n=1 mod 4 -> Theorem 9, n=3 mod 4 -> Theorem 10; together
    these cover every odd n>=5)."""
    vals = [l, m, n]
    if vals.count(2) >= 2:
        rest = vals[:]
        rest.remove(2)
        rest.remove(2)
        third = rest[0] if rest else 2
        if third % 2 == 1 and third >= 5:
            stmt = THEOREM_9 if third % 4 == 1 else THEOREM_10
            return {'solution': None, 'colorable': False, 'colours': None,
                    'colours_tested': 0, 'theorem': stmt}
    return None


THEOREM_CHECKS = [theorem_3, theorem_4, theorem_5, theorem_6,
                   theorem_9_10]  # theorem_7a, theorem_7b, theorem_7c disabled -- see comment above


def check_theorems(l, m, n):
    """Try every proven theorem, in citation order; return the first result
    that applies (None if none do). Trusted as proven -- not re-verified by
    the solver.

    theorem_k1mn runs first and unconditionally, since it is unaffected by
    the issue below. After that, K(2,2,n)-shaped graphs (two of the three
    parts both exactly 2) skip every remaining theorem and fall through to
    a real solve: Theorems 3, 5, 7a, 7b, and 7c were all found to overshoot
    the true minimum by exactly 1 on this family (e.g. K(2,2,6): Theorem 7b
    claims w=9, the real solved minimum is 8) -- see the comment above
    Theorem 7's definitions. This is a targeted guard, not a blanket
    disable: Theorem 3 and Theorem 5 stay active for every other shape
    they match (e.g. K(3,4,7), K(3,5,16)), where they're unverified but
    unchallenged.
    """
    result = theorem_k1mn(l, m, n)
    if result is not None:
        return result
    if [l, m, n].count(2) >= 2:
        # Only theorem_9_10 (the not-colorable claims) has been verified
        # correct on this shape (checked K(2,2,5) against a real solve,
        # genuinely unsat up to the 2*delta cap). theorem_3 and theorem_5
        # overshoot here, so they're skipped -- everything else in
        # THEOREM_CHECKS never matches a double-2 graph anyway.
        return theorem_9_10(l, m, n)
    for fn in THEOREM_CHECKS:
        result = fn(l, m, n)
        if result is not None:
            return result
    return None

## 4. Run

Build K(l, m, n); `check_theorems` (section 3b) gets first look, then
`interval_coloring_optimize` either builds the theorem-proven solution
directly (`fixed_k`, a plain `Solver()` check) or runs the full
`Optimize()`/`minimize()` search. Both independent verification checks from
section 2 still run afterward -- except `verify_minimum`, which is skipped
whenever a theorem already proved minimality (proven theorems are trusted,
not re-verified by z3).

In [17]:
l,m,n = 1,2,3

In [18]:
G, gname, _ = build_graph(l, m, n)

thm = check_theorems(l, m, n)
if thm is not None:
    if thm['colorable']:
        print('Theorem check for %s: colorable with %d colours' % (gname, thm['colours']))
    else:
        print('Theorem check for %s: NOT colorable -- skipping solver' % gname)
    print(thm['theorem'])
    print()

t1 = time.time()
if thm is not None and not thm['colorable']:
    res, progress = thm, []
elif thm is not None and thm['colorable']:
    print('Building the solution directly at %d colours (no search, no minimize()).' % thm['colours'])
    print()
    res, progress = interval_coloring_optimize(G, fixed_k=thm['colours'])
    res['theorem'] = thm['theorem']
else:
    print('Solving %s with Optimize()/minimize (BitVec + symmetry-breaking)' % gname)
    print()
    res, progress = interval_coloring_optimize(G)
t2 = time.time()
runtime = t2 - t1

if res['colorable']:
    print('solution (in %.4fs):' % runtime)
    print(res['solution'])
    print()
    print('Minimum colours for an interval colouring: %d' % res['colours'])
    print('  (max degree Delta = %d is a hard lower bound)' % res['delta'])
    if thm is not None:
        print('Colours proven by theorem -- built directly, no minimize() search (lower=upper=%d)' % res['colours'])
    else:
        print('Optimize() bounds after check(): lower=%d upper=%d  (equal => proven optimal)'
              % (res['lower'], res['upper']))

    valid = verify_interval(res['solution'])
    print('verify_interval (pure python, no z3): %s' % valid)

    if thm is not None:
        is_min = None
        print('verify_minimum: skipped -- minimality already proven by %s' % thm['theorem'])
    else:
        is_min = verify_minimum(G, res)
        if is_min is None:
            print('verify_minimum: N/A (colours already equal to the hard lower bound Delta)')
        else:
            print('verify_minimum (independent Solver() proves colours-1 is unsat): %s' % is_min)
            assert is_min, 'Optimize() claimed a non-minimal optimum -- investigate!'
    assert valid, 'Optimize() returned an invalid interval colouring -- investigate!'
elif thm is not None:
    print('No interval colouring exists for %s (proven by theorem, any k)' % gname)
    print(thm['theorem'])
else:
    print('No interval colouring found with up to %d colours (%.4fs)' % (res['max_colors'], runtime))

Theorem check for k_1_2_3: colorable with 5 colours
Theorem 1: gcd(m+1, n+1) = 1 => K(1,m,n) is colorable, w = m+n

Building the solution directly at 5 colours (no search, no minimize()).

solution (in 0.1237s):
{'0 1': 1, '0 2': 0, '0 3': 2, '0 4': 3, '0 5': 4, '1 3': 3, '1 4': 4, '1 5': 2, '2 3': 1, '2 4': 2, '2 5': 3}

Minimum colours for an interval colouring: 5
  (max degree Delta = 5 is a hard lower bound)
Colours proven by theorem -- built directly, no minimize() search (lower=upper=5)
verify_interval (pure python, no z3): True
verify_minimum: skipped -- minimality already proven by Theorem 1: gcd(m+1, n+1) = 1 => K(1,m,n) is colorable, w = m+n


## 4b. Optimize() progress -- watching the bound tighten

There's no per-`k` loop here to hang a stats table off of -- `Optimize()` does
the whole search inside one `check()` call. The equivalent visibility comes
from `Optimize.set_on_model()`, which z3 calls every time it finds a new
improving model on the way to the optimum (`interval_coloring_optimize` logs
`(time, colours)` at each of those improvements as `progress`). The chart
below shows the objective -- number of colours -- stepping down over
wall-clock time as `Optimize()` tightens the bound, until it lands on (and
then spends the remaining time proving) the optimum.

In [15]:
import matplotlib.pyplot as plt

print('%-10s %-8s' % ('time(s)', 'colours'))
for p in progress:
    print('%-10.4f %-8d' % (p['time'], p['colours']))

if progress:
    times = [p['time'] for p in progress] + [runtime]
    colours = [p['colours'] for p in progress] + [progress[-1]['colours']]

    plt.figure(figsize=(8, 4))
    plt.step(times, colours, where='post', marker='o', color='#1f77b4')
    plt.xlabel('time (s)')
    plt.ylabel('colours (current best M + 1)')
    plt.title('Optimize() bound tightening for %s' % gname)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    if thm is not None:
        print('No improving models recorded (decided by theorem -- no minimize() search ran).')
    else:
        print('No improving models recorded (instance was immediately unsat).')

time(s)    colours 
No improving models recorded (decided by theorem -- no minimize() search ran).


## 5. Draw the coloured graph

Same row-layout drawing as `sat_unsat.ipynb`: nodes coloured by part (same row
= same colour), edges coloured + labelled with their colour number.

In [16]:
# Draw the coloured graph: nodes coloured by part (same row = same colour).
# Layout rules:
#   * every part sits on its own horizontal line,
#   * the part with the FEWEST nodes is pinned to the TOP (2 nodes on top),
#   * the part with the MOST nodes is pinned to the BOTTOM,
#   * generous spacing + spline routing so nodes never overlap and edges
#     curve AROUND nodes instead of crossing over them,
#   * edges coloured and labelled with their colour number.
if not res['colorable']:
    print('No colouring to draw for %s.' % gname)
else:
    sol = res['solution']
    sizes = [int(t) for t in gname.split('_')[1:]]      # 'k_2_3_4' -> [2, 3, 4]

    # >>> CHANGE NODE COLOURS HERE (one per row/part) <<<
    part_fill = ['lightsalmon', 'lightblue', 'lightgreen', 'khaki', 'plum']
    # >>> CHANGE EDGE COLOURS HERE (indexed by each edge's colour number) <<<
    edge_palette = ['red', 'blue', 'green', 'purple', 'orange', 'brown',
                    'cyan', 'magenta', 'darkgreen', 'gray', 'gold', 'navy']

    # node index range for each ORIGINAL part: part p owns nodes [start, start+size)
    starts, node = [], 0
    for size in sizes:
        starts.append(node)
        node += size
    part_nodes_by_part = [[str(starts[p] + i) for i in range(sizes[p])]
                          for p in range(len(sizes))]

    # vertical order of the rows: smallest part on TOP, largest part on BOTTOM
    row_order = sorted(range(len(sizes)), key=lambda p: sizes[p])
    n_rows = len(row_order)

    # spacing so nodes never overlap and edges have room to route AROUND nodes
    G.graph_attr['nodesep'] = '0.6'    # horizontal gap between nodes in a row
    G.graph_attr['ranksep'] = '1.2'    # vertical gap between rows (room for splines)
    G.graph_attr['splines'] = 'true'   # draw edges as curves that avoid nodes

    for row_pos, p in enumerate(row_order):
        part_nodes = part_nodes_by_part[p]
        for nd in part_nodes:
            n = G.get_node(nd)
            n.attr['style'] = 'filled'
            n.attr['fillcolor'] = part_fill[p % len(part_fill)]
        # top row -> rank 'min', bottom row -> rank 'max', middle rows -> 'same'
        rank = 'min' if row_pos == 0 else ('max' if row_pos == n_rows - 1 else 'same')
        sub = G.add_subgraph(part_nodes, name='row_%d' % row_pos, rank=rank)
        # invisible chain keeps the row in increasing left-to-right order
        for a, b in zip(part_nodes, part_nodes[1:]):
            sub.add_edge(a, b, style='invis')

    # colour AND number each edge
    for e_str, color in sol.items():
        u, v = e_str.split(' ')
        e = G.get_edge(u, v)
        c = edge_palette[color % len(edge_palette)]
        e.attr['color'] = c
        e.attr['fontcolor'] = c
        e.attr['label'] = str(color)

    from IPython.display import Image, display
    # prog='dot' forces the layered engine that stacks the rows.
    # (pygraphviz's draw() defaults to neato, which would scramble the rows.)
    display(Image(G.draw(prog='dot', format='png')))   # show inline (NOT saved to disk)

No colouring to draw for k_1_2_2.


## 6. Save graph + matrix to HTML

Appends a section to `solutions_optimize.html` -- kept separate from
`sat_unsat.ipynb`'s `solutions.html` so the two notebooks' saved runs don't
mix. No `solutions.txt` logging in this notebook.

In [ ]:
import base64

png_bytes = G.draw(prog='dot', format='png')
out_path  = 'solutions_optimize.html'

html = ['<section style="font-family:monospace;text-align:center;'
        'max-width:800px;margin:24px auto;'
        'border-bottom:1px solid #ccc;padding-bottom:16px">']
html.append('<h2>%s</h2>' % gname)

html.append('<div>colourable: <b>%s</b></div>' % ('yes' if res['colorable'] else 'no'))
if res.get('theorem'):
    html.append('<div>decided by: <b>%s</b></div>' % res['theorem'])
if res['colorable']:
    html.append('<div>colours used: <b>%d</b></div>' % res['colours'])
    html.append('<div>runtime: %.3f s</div>' % runtime)
    vm_str = 'skipped (theorem-proven)' if thm is not None else str(is_min)
    html.append('<div>verify_interval: <b>%s</b> &nbsp; verify_minimum: <b>%s</b></div>'
                % (valid, vm_str))
    # solution matrix
    html.append('<pre style="background:#f6f6f6;padding:8px;'
                'display:inline-block;text-align:left">%s</pre>'
                % render_matrix(gname, G, res['solution']))
    # image
    b64 = base64.b64encode(png_bytes).decode('ascii')
    html.append('<div><img src="data:image/png;base64,%s" '
                'style="max-width:640px;border:1px solid #ddd"></div>' % b64)
html.append('</section>')

with open(out_path, 'a') as f:
    f.write('\n'.join(html) + '\n')
print('saved image + matrix ->', out_path)